In [1]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

import evidently
from evidently.metric_preset import DataDriftPreset
from evidently.report import Report

# ---- Accès au paquet local `app` ----
PROJECT_ROOT = Path(r"C:\Users\suean\OneDrive\Desktop\tom\OPCL2\P8")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.features_optiweb import apply_eda

print("Python:", sys.executable)
print("evidently:", evidently.__version__)

# ---- Définition des features TOP20 / TOP40 (identiques à ton backend) ----

TOP20_FEATURES = [
    "PAYMENT_RATE",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "DAYS_BIRTH",
    "AMT_ANNUITY",
    "APPROVED_CNT_PAYMENT_MEAN",
    "DAYS_ID_PUBLISH",
    "INSTAL_DPD_MEAN",
    "AMT_CREDIT",
    "INSTAL_AMT_PAYMENT_SUM",
    "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED_PERC",
    "DAYS_REGISTRATION",
    "PREV_CNT_PAYMENT_MEAN",
    "DAYS_EMPLOYED",
    "ACTIVE_DAYS_CREDIT_MAX",
    "INSTAL_DAYS_ENTRY_PAYMENT_MAX",
    "CODE_GENDER",
    "BURO_DAYS_CREDIT_MAX",
]

TOP40_FEATURES = TOP20_FEATURES + [
    "ANNUITY_INCOME_PERC",
    "INCOME_CREDIT_PERC",
    "ACTIVE_DAYS_CREDIT_ENDDATE_MIN",
    "REGION_POPULATION_RELATIVE",
    "DAYS_LAST_PHONE_CHANGE",
    "ACTIVE_DAYS_CREDIT_ENDDATE_MEAN",
    "BURO_DAYS_CREDIT_ENDDATE_MAX",
    "INSTAL_PAYMENT_DIFF_MEAN",
    "PREV_APP_CREDIT_PERC_MEAN",
    "BURO_AMT_CREDIT_SUM_DEBT_MEAN",
    "BURO_AMT_CREDIT_SUM_MEAN",
    "INSTAL_DBD_SUM",
    "POS_MONTHS_BALANCE_MAX",
    "PREV_APP_CREDIT_PERC_MIN",
    "NAME_FAMILY_STATUS_Married",
    "CC_CNT_DRAWINGS_ATM_CURRENT_MEAN",
    "APPROVED_AMT_ANNUITY_MEAN",
    "INSTAL_AMT_PAYMENT_MIN",
    "INSTAL_DAYS_ENTRY_PAYMENT_MEAN",
    "APPROVED_DAYS_DECISION_MAX",
]


Python: c:\Users\suean\OneDrive\Desktop\tom\OPCL2\P8\.venv\Scripts\python.exe
evidently: 0.6.7


In [ ]:
# Limiter le nombre de lignes pour tester (augmente plus tard si tu veux)
NROWS = None   # commence à 50_000 ou 100_000 ; mets None plus tard

print("Reconstruction des features avec apply_eda(...)")

X_train, y_train, X_test, test_ids = apply_eda(
    nrows=NROWS,
    nan_as_category=True,
)

print("Shapes :")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


Reconstruction des features avec apply_eda(...)
Shapes :
X_train: (99997, 755)
X_test : (48744, 755)


In [3]:
# On convertit en DataFrame « plat » pour Evidently
df_train = X_train.copy()
df_test  = X_test.copy()

# Option : sous-échantillonnage pour Evidently (réduit la taille mémoire)
# Ici 50k max par dataset, ajuste selon ta RAM
MAX_ROWS = 50_000

if len(df_train) > MAX_ROWS:
    df_train = df_train.sample(n=MAX_ROWS, random_state=42)

if len(df_test) > MAX_ROWS:
    df_test = df_test.sample(n=MAX_ROWS, random_state=43)

print("df_train used for drift:", df_train.shape)
print("df_test  used for drift:", df_test.shape)

# --- Slices TOP20 / TOP40 (on garde juste les colonnes dispo) ---
cols20 = [c for c in TOP20_FEATURES if c in df_train.columns]
cols40 = [c for c in TOP40_FEATURES if c in df_train.columns]

ref20 = df_train[cols20].copy()
cur20 = df_test[cols20].copy()

ref40 = df_train[cols40].copy()
cur40 = df_test[cols40].copy()

ref20.shape, cur20.shape, ref40.shape, cur40.shape


df_train used for drift: (50000, 755)
df_test  used for drift: (48744, 755)


((50000, 20), (48744, 20), (50000, 40), (48744, 40))

In [4]:
os.makedirs("reports", exist_ok=True)

# Report TOP20
report_top20 = Report(metrics=[
    DataDriftPreset()
])

report_top20.run(
    reference_data=ref20,
    current_data=cur20,
)

report_top20.save_json("reports/drift_top20.json")
print("Saved reports/drift_top20.json")

# Report TOP40
report_top40 = Report(metrics=[
    DataDriftPreset()
])

report_top40.run(
    reference_data=ref40,
    current_data=cur40,
)

report_top40.save_json("reports/drift_top40.json")
print("Saved reports/drift_top40.json")


Saved reports/drift_top20.json
Saved reports/drift_top40.json


In [5]:
def find_first(key, obj):
    """Recherche récursive d'une clé dans un dict/list Evidently."""
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_first(key, v)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_first(key, v)
            if r is not None:
                return r
    return None


def summarize_drift_json(json_path: Path, prefix: str, n_top: int = 40):
    rep_dict = json.loads(json_path.read_text(encoding="utf-8"))

    drift_by_columns = find_first("drift_by_columns", rep_dict) or {}
    number_of_columns = find_first("number_of_columns", rep_dict)
    number_of_drifted = find_first("number_of_drifted_columns", rep_dict)
    share_drifted     = find_first("share_of_drifted_columns", rep_dict)
    dataset_drift     = find_first("dataset_drift", rep_dict)

    rows = []
    for name, info in drift_by_columns.items():
        rows.append({
            "column": name,
            "drift_detected": bool(info.get("drift_detected")),
            "stattest": info.get("stattest_name"),
            "drift_score": info.get("drift_score"),
            "p_value": info.get("p_value"),
            "current_small_hist": info.get("current_small_hist"),
            "ref_small_hist": info.get("ref_small_hist"),
        })
    df = pd.DataFrame(rows)
    df_sorted = df.sort_values(
        ["drift_detected", "drift_score"],
        ascending=[False, False],
        na_position="last",
    )

    # fichiers
    full_csv = Path("reports") / f"{prefix}_drift_columns_full.csv"
    top_csv  = Path("reports") / f"{prefix}_drift_columns_top{n_top}.csv"
    md_file  = Path("reports") / f"{prefix}_drift_summary.md"

    df.to_csv(full_csv, index=False)
    top = df_sorted.head(n_top).copy()
    top.to_csv(top_csv, index=False)

    # résumé
    n_cols = number_of_columns if number_of_columns is not None else len(df)
    n_drift = number_of_drifted if number_of_drifted is not None else int(df["drift_detected"].sum())
    share = share_drifted if share_drifted is not None else (n_drift / n_cols if n_cols else 0.0)
    flag = dataset_drift if dataset_drift is not None else (share > 0.5)

    lines = []
    lines.append(f"# Drift – Conclusion rapide ({prefix})\n")
    lines.append(f"- Colonnes totales : **{n_cols}**")
    lines.append(f"- Colonnes en drift : **{n_drift}**  (part: **{share:.2%}**)  → Dataset drift **{'DETECTÉ' if flag else 'NON détecté'}**")

    if not top.empty:
        lines.append(f"\n**Top {min(n_top, len(top))} colonnes dérivées (drift_score Evidently)** :")
        for i, r in top.iterrows():
            score = r["drift_score"]
            st    = r["stattest"] or "n/a"
            lines.append(f"  - `{r['column']}` — score={score:.3f} — test={st}")

    lines.append("\n**Recommandations** :")
    if share >= 0.25:
        lines.append("- Part de colonnes dérivées élevée → considérer un **retrain** / recalibrage.")
    elif share > 0.1:
        lines.append("- Drift modéré → **surveiller** et vérifier les perfs en ligne.")
    else:
        lines.append("- Drift faible → **rien d’urgent** ; garder un œil sur les variables du top.")

    summary_md = "\n".join(lines)
    md_file.write_text(summary_md, encoding="utf-8")

    print(f"[{prefix}] résumé écrit dans {md_file}")
    print(summary_md[:400], "...\n")  # petit aperçu


# --- Appliquer aux deux rapports ---
summarize_drift_json(Path("reports/drift_top20.json"), prefix="top20", n_top=40)
summarize_drift_json(Path("reports/drift_top40.json"), prefix="top40", n_top=40)

print("Fichiers générés dans ./reports")


[top20] résumé écrit dans reports\top20_drift_summary.md
# Drift – Conclusion rapide (top20)

- Colonnes totales : **20**
- Colonnes en drift : **7**  (part: **35.00%**)  → Dataset drift **NON détecté**

**Top 20 colonnes dérivées (drift_score Evidently)** :
  - `PAYMENT_RATE` — score=0.572 — test=Wasserstein distance (normed)
  - `INSTAL_DAYS_ENTRY_PAYMENT_MAX` — score=0.352 — test=Wasserstein distance (normed)
  - `EXT_SOURCE_1` — score=0.248 — test=W ...

[top40] résumé écrit dans reports\top40_drift_summary.md
# Drift – Conclusion rapide (top40)

- Colonnes totales : **40**
- Colonnes en drift : **11**  (part: **27.50%**)  → Dataset drift **NON détecté**

**Top 40 colonnes dérivées (drift_score Evidently)** :
  - `PAYMENT_RATE` — score=0.572 — test=Wasserstein distance (normed)
  - `INSTAL_DAYS_ENTRY_PAYMENT_MEAN` — score=0.404 — test=Wasserstein distance (normed)
  - `INSTAL_DAYS_ENTRY_PAYMENT_MAX` — s ...

Fichiers générés dans ./reports


In [6]:
# ============================================
# Rapport HTML Evidently – TOP 40 features
# ============================================
from pathlib import Path
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# même liste que dans ton backend
TOP40_FEATURES = [
    "PAYMENT_RATE",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "DAYS_BIRTH",
    "AMT_ANNUITY",
    "APPROVED_CNT_PAYMENT_MEAN",
    "DAYS_ID_PUBLISH",
    "INSTAL_DPD_MEAN",
    "AMT_CREDIT",
    "INSTAL_AMT_PAYMENT_SUM",
    "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED_PERC",
    "DAYS_REGISTRATION",
    "PREV_CNT_PAYMENT_MEAN",
    "DAYS_EMPLOYED",
    "ACTIVE_DAYS_CREDIT_MAX",
    "INSTAL_DAYS_ENTRY_PAYMENT_MAX",
    "CODE_GENDER",
    "BURO_DAYS_CREDIT_MAX",
    "ANNUITY_INCOME_PERC",
    "INCOME_CREDIT_PERC",
    "ACTIVE_DAYS_CREDIT_ENDDATE_MIN",
    "REGION_POPULATION_RELATIVE",
    "DAYS_LAST_PHONE_CHANGE",
    "ACTIVE_DAYS_CREDIT_ENDDATE_MEAN",
    "BURO_DAYS_CREDIT_ENDDATE_MAX",
    "INSTAL_PAYMENT_DIFF_MEAN",
    "PREV_APP_CREDIT_PERC_MEAN",
    "BURO_AMT_CREDIT_SUM_DEBT_MEAN",
    "BURO_AMT_CREDIT_SUM_MEAN",
    "INSTAL_DBD_SUM",
    "POS_MONTHS_BALANCE_MAX",
    "PREV_APP_CREDIT_PERC_MIN",
    "NAME_FAMILY_STATUS_Married",
    "CC_CNT_DRAWINGS_ATM_CURRENT_MEAN",
    "APPROVED_AMT_ANNUITY_MEAN",
    "INSTAL_AMT_PAYMENT_MIN",
    "INSTAL_DAYS_ENTRY_PAYMENT_MEAN",
    "APPROVED_DAYS_DECISION_MAX",
]

# sécurité: on ne garde que les colonnes vraiment présentes
top40_cols = [c for c in TOP40_FEATURES if c in X_train.columns]
print(f"{len(top40_cols)} colonnes utilisées sur 40 :", top40_cols[:5], "...")

ref_top40 = X_train[top40_cols].copy()
cur_top40 = X_test[top40_cols].copy()

# créer le dossier reports *depuis datadrift/*
Path("reports").mkdir(exist_ok=True)

# rapport Evidently
report_top40 = Report(metrics=[DataDriftPreset()])
report_top40.run(reference_data=ref_top40, current_data=cur_top40)

html_path = Path("reports") / "drift_top40.html"
report_top40.save_html(str(html_path))

print(f"Rapport HTML TOP40 écrit dans: {html_path.resolve()}")


40 colonnes utilisées sur 40 : ['PAYMENT_RATE', 'EXT_SOURCE_3', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'DAYS_BIRTH'] ...
Rapport HTML TOP40 écrit dans: C:\Users\suean\OneDrive\Desktop\tom\OPCL2\P8\reports\drift_top40.html
